
## Folder 06 — File 02: Orphan Record Handling:


This notebook will:

- Identify orphan records.
- Separate null foreign keys from missing parent references.
- Store orphan records in quarantine.
- Preserve the original source records.
- Recheck orphan records after parent data arrives.
- Identify records that are ready for reprocessing.
- Maintain orphan history.

**Example:** 

An order references customer CUST999, but that customer has not arrived in Silver yet. Instead of deleting the order, quarantine it. When the customer arrives, identify the order as eligible for reprocessing.


### orphan record:


An **orphan record** is a record in a `child table` that references a record in a `parent table` that does not exist.


**In simple words:**

A **foreign key** value exists in `one table`, but the **matching record** is missing in the `related table`.

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"
SCHEMA = "ecommerce_silver"

ORPHAN_SOURCE_TABLE = (
    f"{CATALOG}.{SCHEMA}.referential_integrity_orphans"
)

ORPHAN_QUARANTINE_TABLE = (
    f"{CATALOG}.{SCHEMA}.orphan_record_quarantine"
)

ORPHAN_HISTORY_TABLE = (
    f"{CATALOG}.{SCHEMA}.orphan_record_history"
)

READY_FOR_REPROCESS_TABLE = (
    f"{CATALOG}.{SCHEMA}.orphan_records_ready_for_reprocessing"
)

RUN_ID = str(uuid.uuid4())

print("Run ID:", RUN_ID)


## 3. Load referential integrity results

if not spark.catalog.tableExists(ORPHAN_SOURCE_TABLE):

    raise Exception(
        "Referential integrity orphan table not found. "
        "Run File 01 first."
    )

orphan_source_df = spark.table(
    ORPHAN_SOURCE_TABLE
)

print("Orphan records loaded:", orphan_source_df.count())

display(orphan_source_df)


## 4. Separate null references and missing parent references

null_reference_df = (
    orphan_source_df
    .filter(
        F.col("invalid_reference_value").isNull()
        | (
            F.trim(
                F.col("invalid_reference_value")
            ) == ""
        )
    )
)

missing_parent_df = (
    orphan_source_df
    .filter(
        F.col("invalid_reference_value").isNotNull()
        & (
            F.trim(
                F.col("invalid_reference_value")
            ) != ""
        )
    )
)

print("Null foreign-key records:", null_reference_df.count())
print("Missing parent reference records:", missing_parent_df.count())


## 5. Add quarantine metadata

def add_quarantine_metadata(df, issue_type):

    return (
        df
        .withColumn(
            "issue_type",
            F.lit(issue_type)
        )
        .withColumn(
            "quarantine_status",
            F.lit("OPEN")
        )
        .withColumn(
            "quarantine_run_id",
            F.lit(RUN_ID)
        )
        .withColumn(
            "quarantined_at",
            F.current_timestamp()
        )
        .withColumn(
            "last_checked_at",
            F.current_timestamp()
        )
    )


null_quarantine_df = add_quarantine_metadata(
    null_reference_df,
    "NULL_FOREIGN_KEY"
)

missing_parent_quarantine_df = add_quarantine_metadata(
    missing_parent_df,
    "MISSING_PARENT_RECORD"
)

combined_quarantine_df = (
    null_quarantine_df
    .unionByName(
        missing_parent_quarantine_df
    )
)

display(combined_quarantine_df)


## 6. Create current orphan quarantine table

(
    combined_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ORPHAN_QUARANTINE_TABLE)
)

print("Current orphan quarantine table saved.")


 ## 7. Append records to orphan history

if spark.catalog.tableExists(ORPHAN_HISTORY_TABLE):

    (
        combined_quarantine_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(ORPHAN_HISTORY_TABLE)
    )

else:

    (
        combined_quarantine_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(ORPHAN_HISTORY_TABLE)
    )

print("Orphan history updated.")


## 8. Load current parent datasets

PARENT_TABLES = {
    "customers": f"{CATALOG}.{SCHEMA}.silver_customers",
    "orders": f"{CATALOG}.{SCHEMA}.silver_orders",
    "products": f"{CATALOG}.{SCHEMA}.silver_products"
}

for table_name in PARENT_TABLES.values():

    if not spark.catalog.tableExists(table_name):

        raise Exception(
            f"Parent table not found: {table_name}"
        )

customers_df = spark.table(
    PARENT_TABLES["customers"]
)

orders_df = spark.table(
    PARENT_TABLES["orders"]
)

products_df = spark.table(
    PARENT_TABLES["products"]
)


## 9. Define relationship lookup configuration

PARENT_LOOKUPS = {

    "orders_to_customers": {
        "parent_df": customers_df,
        "parent_key": "customer_id"
    },

    "order_items_to_orders": {
        "parent_df": orders_df,
        "parent_key": "order_id"
    },

    "order_items_to_products": {
        "parent_df": products_df,
        "parent_key": "product_id"
    },

    "payments_to_orders": {
        "parent_df": orders_df,
        "parent_key": "order_id"
    },

    "shipments_to_orders": {
        "parent_df": orders_df,
        "parent_key": "order_id"
    },

    "inventory_to_products": {
        "parent_df": products_df,
        "parent_key": "product_id"
    }
}


## 10. Recheck missing parent references


def find_resolved_orphans(orphan_df):

    resolved_dataframes = []
    unresolved_dataframes = []

    for relationship_name, config in PARENT_LOOKUPS.items():

        relationship_df = (
            orphan_df
            .filter(
                F.col("relationship") == relationship_name
            )
        )

        if relationship_df.limit(1).count() == 0:
            continue

        parent_df = config["parent_df"]
        parent_key = config["parent_key"]

        parent_keys_df = (
            parent_df
            .select(
                F.col(parent_key).cast("string").alias(
                    "_available_parent_key"
                )
            )
            .filter(
                F.col("_available_parent_key").isNotNull()
            )
            .distinct()
        )

        join_condition = (
            F.col("orphan.invalid_reference_value")
            == F.col("parent._available_parent_key")
        )

        resolved_df = (
            relationship_df.alias("orphan")
            .join(
                parent_keys_df.alias("parent"),
                join_condition,
                "inner"
            )
            .select("orphan.*")
        )

        unresolved_df = (
            relationship_df.alias("orphan")
            .join(
                parent_keys_df.alias("parent"),
                join_condition,
                "left_anti"
            )
            .select("orphan.*")
        )

        resolved_dataframes.append(resolved_df)
        unresolved_dataframes.append(unresolved_df)

    if resolved_dataframes:

        resolved_combined_df = resolved_dataframes[0]

        for next_df in resolved_dataframes[1:]:

            resolved_combined_df = (
                resolved_combined_df.unionByName(next_df)
            )

    else:

        resolved_combined_df = (
            orphan_df.limit(0)
        )

    if unresolved_dataframes:

        unresolved_combined_df = unresolved_dataframes[0]

        for next_df in unresolved_dataframes[1:]:

            unresolved_combined_df = (
                unresolved_combined_df.unionByName(next_df)
            )

    else:

        unresolved_combined_df = (
            orphan_df.limit(0)
        )

    return resolved_combined_df, unresolved_combined_df



## 11. Identify records ready for reprocessing

current_orphans_df = spark.table(
    ORPHAN_QUARANTINE_TABLE
)

non_null_orphans_df = (
    current_orphans_df
    .filter(
        F.col("issue_type") == "MISSING_PARENT_RECORD"
    )
)

resolved_df, unresolved_df = find_resolved_orphans(
    non_null_orphans_df
)

ready_for_reprocessing_df = (
    resolved_df
    .withColumn(
        "reprocessing_status",
        F.lit("READY_FOR_REPROCESSING")
    )
    .withColumn(
        "resolved_at",
        F.current_timestamp()
    )
)

still_unresolved_df = (
    unresolved_df
    .withColumn(
        "reprocessing_status",
        F.lit("WAITING_FOR_PARENT")
    )
)

display(ready_for_reprocessing_df)

display(still_unresolved_df)


## 12. Save ready-for-reprocessing records

(
    ready_for_reprocessing_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(READY_FOR_REPROCESS_TABLE)
)

print(
    "Ready records saved:",
    READY_FOR_REPROCESS_TABLE
)


## 13. Generate orphan handling summary

handling_summary_df = (
    current_orphans_df
    .groupBy(
        "relationship",
        "issue_type",
        "quarantine_status"
    )
    .agg(
        F.count("*").alias("total_records")
    )
    .withColumn(
        "summary_generated_at",
        F.current_timestamp()
    )
    .orderBy(
        "relationship",
        "issue_type"
    )
)

display(handling_summary_df)


## 14. Display reprocessing statistics

reprocessing_summary_df = (
    ready_for_reprocessing_df
    .agg(
        F.count("*").alias("ready_records")
    )
    .withColumn(
        "waiting_records",
        F.lit(unresolved_df.count())
    )
    .withColumn(
        "null_reference_records",
        F.lit(null_reference_df.count())
    )
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
)

display(reprocessing_summary_df)


## 15. Final validation

print("=" * 65)
print("ORPHAN RECORD HANDLING SUMMARY")
print("=" * 65)

print(
    "Total quarantined:",
    current_orphans_df.count()
)

print(
    "Null references:",
    null_reference_df.count()
)

print(
    "Missing parent references:",
    missing_parent_df.count()
)

print(
    "Ready for reprocessing:",
    ready_for_reprocessing_df.count()
)

print(
    "Still waiting for parent:",
    unresolved_df.count()
)

print("=" * 65)

print("Quarantine table:", ORPHAN_QUARANTINE_TABLE)
print("History table:", ORPHAN_HISTORY_TABLE)
print("Reprocessing table:", READY_FOR_REPROCESS_TABLE)

# COMMAND ----------
